In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_DIR = PROJECT_ROOT
FEATURE_FILE = OUTPUT_ROOT / "features" / "Master_Features_2015_2025.csv"
OUTPUT_DIR = OUTPUT_ROOT / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ML_FILE = OUTPUT_ROOT / "cpcb" / "ML_Dataset_2015_2025.csv"


In [ ]:
features = pd.read_csv(FEATURE_FILE)

features.head()

In [ ]:
FEATURES = [

    "Blue",

    "Green",

    "Red",

    "VIIRS",

    "LULC"

]

In [ ]:
prediction_data = features.copy()

prediction_data[FEATURES] = prediction_data[FEATURES].fillna(

    prediction_data[FEATURES].median()

)

ML Dataset Load

In [ ]:
ML_FILE = OUTPUT_ROOT / "cpcb" / "ML_Dataset_2015_2025.csv"

ml = pd.read_csv(ML_FILE)

ml.head()

In [ ]:
TARGETS = [

    "_do",

    "bod",

    "cod",

    "ph_gen",

    "ec_gen",

    "temp"

]

dataset function

In [ ]:
def prepare_dataset(df, target):

    data = df[FEATURES + [target]].copy()

    data = data.dropna(subset=[target])

    data[FEATURES] = data[FEATURES].fillna(

        data[FEATURES].median()

    )

    return data

In [ ]:
best_models = {

    "_do": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "bod": CatBoostRegressor(
        iterations=300,
        random_state=42,
        verbose=False
    ),

    "cod": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "ph_gen": CatBoostRegressor(
        iterations=300,
        random_state=42,
        verbose=False
    ),

    "ec_gen": CatBoostRegressor(
        iterations=300,
        random_state=42,
        verbose=False
    ),

    "temp": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

}

In [ ]:
prediction_columns = {

    "_do": "Predicted_DO",

    "bod": "Predicted_BOD",

    "cod": "Predicted_COD",

    "ph_gen": "Predicted_pH",

    "ec_gen": "Predicted_EC",

    "temp": "Predicted_Temperature"

}

for target, model in best_models.items():

    print(f"Training model for {target}")

    dataset = prepare_dataset(ml, target)

    X = dataset[FEATURES]

    y = dataset[target]

    model.fit(X, y)

    prediction_data[prediction_columns[target]] = model.predict(

        prediction_data[FEATURES]

    )

print("Prediction Completed")

In [ ]:
prediction_data.to_csv(

    OUTPUT_DIR / "Water_Quality_Predictions.csv",

    index=False

)

print("CSV Saved Successfully.")

In [ ]:
ML_FILE = OUTPUT_ROOT / "cpcb" / "ML_Dataset_2015_2025.csv"

In [ ]:
best_model_names = {
    "_do": "Random Forest",
    "bod": "CatBoost",
    "cod": "Random Forest",
    "ph_gen": "CatBoost",
    "ec_gen": "CatBoost",
    "temp": "Random Forest"
}

In [ ]:
def get_model(model_name):

    if model_name == "Random Forest":
        return RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )

    elif model_name == "CatBoost":
        return CatBoostRegressor(
            iterations=300,
            random_state=42,
            verbose=False
        )

    elif model_name == "XGBoost":
        return XGBRegressor(
            n_estimators=300,
            random_state=42,
            objective="reg:squarederror"
        )

    elif model_name == "LightGBM":
        return LGBMRegressor(
            n_estimators=300,
            random_state=42
        )

In [ ]:
#prediction loop
for target in TARGETS:

    model = get_model(best_model_names[target])

    dataset = prepare_dataset(ml, target)

    X = dataset[FEATURES]
    y = dataset[target]

    model.fit(X, y)

    prediction_data[f"Predicted_{target}"] = model.predict(
        prediction_data[FEATURES]
    )

In [ ]:
prediction_columns = {
    "_do": "Predicted_DO",
    "bod": "Predicted_BOD",
    "cod": "Predicted_COD",
    "ph_gen": "Predicted_pH",
    "ec_gen": "Predicted_EC",
    "temp": "Predicted_Temperature"
}

for target in TARGETS:

    print(f"\nPredicting {target}...")

    model = get_model(best_model_names[target])

    dataset = prepare_dataset(ml, target)

    X_train = dataset[FEATURES]
    y_train = dataset[target]

    model.fit(X_train, y_train)

    prediction_data[prediction_columns[target]] = model.predict(
        prediction_data[FEATURES]
    )

print("\n✅ All Predictions Completed!")

In [ ]:
OUTPUT_FILE = OUTPUT_DIR / "Water_Quality_Predictions.csv"

prediction_data.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"✅ File Saved Successfully!\n\n{OUTPUT_FILE}")

In [ ]:
prediction_data.head()

In [ ]:
prediction_data.columns

In [ ]:
prediction_data.shape

columns ke duplication bngye

In [ ]:
duplicate_cols = [
    "Predicted_do",
    "Predicted_bod",
    "Predicted_cod",
    "Predicted_ph_gen",
    "Predicted_ec_gen",
    "Predicted_temp"
]

prediction_data.drop(
    columns=duplicate_cols,
    inplace=True,
    errors="ignore"
)

In [ ]:
prediction_data.describe()

In [ ]:
prediction_data.isnull().sum()

In [ ]:
# ============================================================
# SAVE FINAL TRAINED MODELS FOR STREAMLIT
# ============================================================

import joblib
from pathlib import Path

# Create models folder
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# Save each trained model
joblib.dump(
    best_models["_do"],
    MODEL_DIR / "DO_RF.joblib"
)

joblib.dump(
    best_models["bod"],
    MODEL_DIR / "BOD_CatBoost.joblib"
)

joblib.dump(
    best_models["cod"],
    MODEL_DIR / "COD_RF.joblib"
)

joblib.dump(
    best_models["ph_gen"],
    MODEL_DIR / "pH_CatBoost.joblib"
)

joblib.dump(
    best_models["ec_gen"],
    MODEL_DIR / "EC_CatBoost.joblib"
)

joblib.dump(
    best_models["temp"],
    MODEL_DIR / "Temperature_RF.joblib"
)

print("✅ All 6 trained models saved successfully!")

print("\nSaved model files:")

for file in MODEL_DIR.iterdir():
    print("  ✓", file)